In [105]:
import os
import glob
import pandas as pd
from functools import reduce

from chain import Chain

In [99]:
csv_folder = "cris_data/"
output_file = "collisions_db.csv"

In [ ]:
# Function to clean column names
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [col.strip().replace(" ", "_") for col in df.columns]
    return df

collision_etl_chain = (Chain()
    .map(lambda file: pd.read_csv(file, skiprows=11)) # load files
    .pipe(lambda dfs: pd.concat(dfs, ignore_index=True)) # concatenate DataFrames
    .pipe(clean_column_names) # clean column names
)

def DataPreprocessor(csv_folder: str, function_chain: Chain):
    def fetch_csv_files(csv_folder):
        return glob.glob(os.path.join(csv_folder, "*.csv"))
    
    csv_files = fetch_csv_files(csv_folder)
    combined_df = function_chain(csv_files)
    return combined_df

combined_df = DataPreprocessor(csv_folder, collision_etl_chain)

In [101]:
combined_df.shape

(197544, 11)

In [102]:
combined_df.head()

,Crash_ID,Crash_Month,Crash_Severity,Crash_Year,Day_of_Week,Hour_of_Day,On_System_Flag,Person_Age,Person_Ethnicity,Person_Gender,Person_Type
0,18039083,1,C - POSSIBLE INJURY,2021,FRIDAY,03:00 - 03:59,Yes,22,H - HISPANIC,1 - MALE,1 - DRIVER
1,18039338,1,N - NOT INJURED,2021,FRIDAY,07:00 - 07:59,No,30,H - HISPANIC,2 - FEMALE,1 - DRIVER
2,18039360,1,N - NOT INJURED,2021,FRIDAY,11:00 - 11:59,No,55,W - WHITE,1 - MALE,5 - DRIVER OF MOTORCYCLE TYPE VEHICLE
3,18039566,1,N - NOT INJURED,2021,FRIDAY,15:00 - 15:59,No,24,W - WHITE,2 - FEMALE,1 - DRIVER
4,18039566,1,N - NOT INJURED,2021,FRIDAY,15:00 - 15:59,No,73,W - WHITE,2 - FEMALE,1 - DRIVER


In [103]:
combined_df['Hour_of_Day'].dtype

<StringDtype(storage='python', na_value=nan)>

In [104]:
# --------------------------
# Extract first hour from range (e.g., "11:00 - 11:59")
# --------------------------
def extract_first_hour(hour_str: str) -> str:
    if pd.isna(hour_str):
        return pd.NA
    try:
        return str(hour_str).split(" - ")[0].strip()
    except:
        return pd.NA

# --------------------------
# Convert 24-hour string "HH:MM" to 12-hour integer
# --------------------------
def hour_24_to_12(hour_str: str) -> str:
    if pd.isna(hour_str):
        return pd.NA
    try:
        hour = int(hour_str.split(":")[0])
    except:
        return pd.NA

    if hour == 0:
        return "12 AM"
    elif hour < 12:
        return f"{hour} AM"
    elif hour == 12:
        return "12 PM"
    else:
        return f"{hour-12} PM"
    
def series_pipeline(*funcs):
    """
    Returns a function that applies a series of transformations to a Series.
    Each function should take a Series and return a Series.
    """
    def apply_pipeline(series: pd.Series) -> pd.Series:
        return reduce(lambda s, f: f(s), funcs, series)
    return apply_pipeline

# Create the pipeline: first extract first hour, then convert to 12-hour format
hour_pipeline = series_pipeline(
    lambda s: s.apply(extract_first_hour),
    lambda s: s.apply(hour_24_to_12)
)

# Apply the series pipeline and assign to the column
combined_df["Hour_of_Day"] = hour_pipeline(combined_df["Hour_of_Day"])
combined_df

,Crash_ID,Crash_Month,Crash_Severity,Crash_Year,Day_of_Week,Hour_of_Day,On_System_Flag,Person_Age,Person_Ethnicity,Person_Gender,Person_Type
0,18039083,1,C - POSSIBLE INJURY,2021,FRIDAY,3 AM,Yes,22,H - HISPANIC,1 - MALE,1 - DRIVER
1,18039338,1,N - NOT INJURED,2021,FRIDAY,7 AM,No,30,H - HISPANIC,2 - FEMALE,1 - DRIVER
2,18039360,1,N - NOT INJURED,2021,FRIDAY,11 AM,No,55,W - WHITE,1 - MALE,5 - DRIVER OF MOTORCYCLE TYPE VEHICLE
3,18039566,1,N - NOT INJURED,2021,FRIDAY,3 PM,No,24,W - WHITE,2 - FEMALE,1 - DRIVER
4,18039566,1,N - NOT INJURED,2021,FRIDAY,3 PM,No,73,W - WHITE,2 - FEMALE,1 - DRIVER
...,...,...,...,...,...,...,...,...,...,...,...
197539,21292092,2,N - NOT INJURED,2026,MONDAY,8 AM,Yes,41,H - HISPANIC,1 - MALE,1 - DRIVER
197540,21292092,2,N - NOT INJURED,2026,MONDAY,8 AM,Yes,33,H - HISPANIC,1 - MALE,2 - PASSENGER/OCCUPANT
197541,21292092,2,N - NOT INJURED,2026,MONDAY,8 AM,Yes,44,H - HISPANIC,2 - FEMALE,1 - DRIVER
197542,21292199,2,N - NOT INJURED,2026,MONDAY,12 PM,No,78,W - WHITE,1 - MALE,1 - DRIVER


In [113]:
def person_ethnicity_formatting(ethnicity_str: str) -> str:
    short_str = ethnicity_str.split(" - ")[1].strip()
    return short_str

def person_gender_formatting(person_gender_str: str) -> str:
    short_str = person_gender_str.split(" - ")[1].strip()
    return short_str

def person_type_formatting(person_type_str: str) -> str:
    short_str = person_type_str.split(" - ")[1].strip()
    return short_str

person_ethicity_pipeline = series_pipeline(
    lambda s: s.apply(person_ethnicity_formatting)
)

person_gender_pipeline = series_pipeline(
    lambda s: s.apply(person_gender_formatting)
)

person_type_pipeline = series_pipeline(
    lambda s: s.apply(person_type_formatting)
)

pipeline_dict = {"Person_Ethnicity": person_ethicity_pipeline,
                 "Person_Gender" : person_gender_pipeline,
                 "Person_Type" : person_type_pipeline
}

for column, pipeline in pipeline_dict.items():
    combined_df[column] = pipeline(combined_df[column])

combined_df

IndexError: list index out of range

In [108]:
filters_chain = (Chain()
    .pipe(lambda df: df.query("`On_System_Flag` == 'No'")) # filter rows where 'On System Flag' is 'No'
    .pipe(lambda df: df.query("`Crash_Severity` in ['A - SUSPECTED SERIOUS INJURY', 'K - FATAL INJURY']"))  # keep Fatal or Serious
)

combined_df = filters_chain(combined_df)
combined_df.to_csv("collisions_db.csv", index=False)
combined_df

,Crash_ID,Crash_Month,Crash_Severity,Crash_Year,Day_of_Week,Hour_of_Day,On_System_Flag,Person_Age,Person_Ethnicity,Person_Gender,Person_Type
341,18063119,1,A - SUSPECTED SERIOUS INJURY,2021,TUESDAY,11 AM,No,22,B - BLACK,1 - MALE,5 - DRIVER OF MOTORCYCLE TYPE VEHICLE
684,18057364,1,K - FATAL INJURY,2021,TUESDAY,3 PM,No,26,H - HISPANIC,1 - MALE,5 - DRIVER OF MOTORCYCLE TYPE VEHICLE
685,18057364,1,K - FATAL INJURY,2021,TUESDAY,3 PM,No,69,H - HISPANIC,2 - FEMALE,1 - DRIVER
904,18062920,1,A - SUSPECTED SERIOUS INJURY,2021,THURSDAY,3 PM,No,26,H - HISPANIC,1 - MALE,5 - DRIVER OF MOTORCYCLE TYPE VEHICLE
905,18062920,1,A - SUSPECTED SERIOUS INJURY,2021,THURSDAY,3 PM,No,38,W - WHITE,1 - MALE,1 - DRIVER
...,...,...,...,...,...,...,...,...,...,...,...
196014,21285498,2,A - SUSPECTED SERIOUS INJURY,2026,FRIDAY,2 PM,No,60,H - HISPANIC,1 - MALE,1 - DRIVER
196035,21265888,2,K - FATAL INJURY,2026,SATURDAY,5 PM,No,62,W - WHITE,2 - FEMALE,1 - DRIVER
196036,21265888,2,K - FATAL INJURY,2026,SATURDAY,5 PM,No,57,H - HISPANIC,2 - FEMALE,4 - PEDESTRIAN
196115,21266118,2,K - FATAL INJURY,2026,SUNDAY,2 AM,No,20,B - BLACK,2 - FEMALE,1 - DRIVER
